In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# describe(), 결측치 표, 상관계수 표를 보기 좋게 만들기 위한 기본 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [2]:
music_df = pd.read_csv('01_data/processed/train.csv')
census_df = pd.read_csv('01_data/processed/acs2015_census_tract_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '01_data/processed/train.csv'

In [ ]:
# location 오타 수정
music_df['location'] = music_df['location'].replace({
    'Nebrasksa': 'Nebraska'
})

In [ ]:
# 원본 데이터 기본 점검
music_raw = music_df.copy()
census_raw = census_df.copy()

print(f"music_df shape: {music_df.shape}")
print(f"census_df shape: {census_df.shape}")

raw_summary_music = pd.DataFrame({
    'dtype': music_df.dtypes.astype(str),
    'missing_cnt': music_df.isna().sum(),
    'missing_ratio(%)': (music_df.isna().mean() * 100).round(2),
    'nunique': music_df.nunique(dropna=False)
}).sort_values(['missing_cnt', 'nunique'], ascending=[False, False])

display(raw_summary_music)

num_cols_music = music_df.select_dtypes(include=np.number).columns.tolist()
cat_cols_music = music_df.select_dtypes(exclude=np.number).columns.tolist()

print("수치형 컬럼:", num_cols_music)
print("범주형 컬럼:", cat_cols_music)

display(music_df[num_cols_music].describe().T)

print("\n[churned 분포]")
display(
    music_df['churned']
    .value_counts(dropna=False)
    .rename_axis('churned')
    .reset_index(name='count')
    .assign(ratio=lambda x: (x['count'] / len(music_df) * 100).round(2))
)

eda_cat_cols = ['location', 'subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries']
for col in eda_cat_cols:
    print(f"\n[{col}]")
    display(
        music_df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name='count')
        .assign(ratio=lambda x: (x['count'] / len(music_df) * 100).round(2))
        .head(20)
    )

In [ ]:
# customer_id 삭제
if 'customer_id' in music_df.columns:
    music_df = music_df.drop('customer_id', axis=1)

In [ ]:
income_missing_before = census_df['Income'].isna().sum()
zero_totalpop_cnt = (census_df['TotalPop'] == 0).sum()

# Census 데이터 전처리
census_df['Income'] = census_df.groupby('State')['Income'].transform(lambda x: x.fillna(x.median()))

# State별 집계
state_stats = census_df.groupby('State').agg({
    'TotalPop': 'sum',
    'Income': 'mean'
}).reset_index()

# 컬럼명 변경
state_stats.columns = ['State', 'State_TotalPop', 'State_AvgIncome']

print(f"Income 결측치(전): {income_missing_before}")
print(f"Income 결측치(후): {census_df['Income'].isna().sum()}")
print(f"TotalPop == 0 개수: {zero_totalpop_cnt}")

print("\n[state_stats 기본 통계]")
display(state_stats.describe().T)

music_states = set(music_df['location'].dropna().unique())
census_states = set(state_stats['State'].dropna().unique())

only_music = sorted(music_states - census_states)
only_census = sorted(census_states - music_states)

print("\nmusic_df에만 있는 location 값")
print(only_music)

print("\nstate_stats에만 있는 State 값")
print(only_census)

In [ ]:
# 데이터 병합 + 병합 상태 점검
final_df = pd.merge(
    music_df,
    state_stats,
    left_on='location',
    right_on='State',
    how='left',
    indicator=True
)

print("병합 결과 요약")
display(
    final_df['_merge']
    .value_counts(dropna=False)
    .rename_axis('_merge')
    .reset_index(name='count')
    .assign(ratio=lambda x: (x['count'] / len(final_df) * 100).round(2))
)

print("\n매칭 실패 location 상위")
display(
    final_df.loc[final_df['_merge'] != 'both', 'location']
    .value_counts(dropna=False)
    .rename_axis('location')
    .reset_index(name='count')
    .head(20)
)

In [ ]:
# 병합 확인용 컬럼 제거
final_df = final_df.drop(['State', '_merge'], axis=1)

In [ ]:
# 수치 데이터 보정
final_df['average_session_length'] = final_df['average_session_length'] / 60
final_df['weekly_unique_songs'] = np.where(
    final_df['weekly_unique_songs'] > final_df['weekly_songs_played'],
    final_df['weekly_songs_played'],
    final_df['weekly_unique_songs']
)

final_df['tenure_days'] = final_df['signup_date'].abs()
final_df = final_df.drop('signup_date', axis=1)

In [ ]:
# 전처리 논리 점검
logic_check = pd.Series({
    'weekly_unique_songs > weekly_songs_played': int((final_df['weekly_unique_songs'] > final_df['weekly_songs_played']).sum()),
    'average_session_length <= 0': int((final_df['average_session_length'] <= 0).sum()),
    'song_skip_rate < 0 or > 1': int(((final_df['song_skip_rate'] < 0) | (final_df['song_skip_rate'] > 1)).sum()),
    'weekly_hours < 0': int((final_df['weekly_hours'] < 0).sum()),
    'tenure_days < 0': int((final_df['tenure_days'] < 0).sum())
}, name='count')

display(logic_check.to_frame())

print("\ntenure_days 요약")
display(final_df['tenure_days'].describe().to_frame().T)

# 가설 EDA용 원본 형태 백업
eda_df = final_df.copy()

In [ ]:
# 문자열 처리
final_df = final_df.drop('location', axis=1, errors='ignore')

cols_to_encode = ['subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries']
final_df = pd.get_dummies(final_df, columns=cols_to_encode, drop_first=True, dtype=int)

In [ ]:
# 최종 데이터 결과 확인
print(f"최종 데이터 형태: {final_df.shape}")
display(final_df.head())

print("\n최종 데이터 기본 통계")
display(final_df.describe().T)

print("\n최종 데이터 결측치 요약")
missing_summary = pd.DataFrame({
    'missing_cnt': final_df.isna().sum(),
    'missing_ratio(%)': (final_df.isna().mean() * 100).round(2),
    'nunique': final_df.nunique(dropna=False)
}).sort_values(['missing_cnt', 'nunique'], ascending=[False, False])

display(missing_summary)

print("\n중복 행 수:", final_df.duplicated().sum())

print("\n타깃(churned) 분포")
display(
    final_df['churned']
    .value_counts(dropna=False)
    .rename_axis('churned')
    .reset_index(name='count')
    .assign(ratio=lambda x: (x['count'] / len(final_df) * 100).round(2))
)

constant_cols = [col for col in final_df.columns if final_df[col].nunique(dropna=False) <= 1]
print("\n상수 컬럼:", constant_cols)

In [ ]:
# 수치형 이상치(IQR) 요약
# 연속형/실수형 중심 컬럼만 선택
iqr_cols = [
    'age', 'tenure_days', 'weekly_hours', 'average_session_length', 'song_skip_rate',
    'weekly_songs_played', 'weekly_unique_songs', 'num_favorite_artists',
    'num_platform_friends', 'num_playlists_created', 'num_shared_playlists',
    'notifications_clicked', 'State_TotalPop', 'State_AvgIncome'
]

outlier_rows = []
for col in iqr_cols:
    q1 = final_df[col].quantile(0.25)
    q3 = final_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    out_cnt = ((final_df[col] < lower) | (final_df[col] > upper)).sum()

    outlier_rows.append({
        'column': col,
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower': lower,
        'upper': upper,
        'outlier_cnt': int(out_cnt),
        'outlier_ratio(%)': round(out_cnt / len(final_df) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values('outlier_ratio(%)', ascending=False)
display(outlier_summary.head(20))

# churned와의 상관계수 미리 보기
corr_with_target = (
    final_df.corr(numeric_only=True)['churned']
    .drop('churned')
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("\nchurned와의 상관계수 상위 변수")
display(corr_with_target.head(20).to_frame('corr_with_churned'))

In [ ]:
# eda_df
# - 원본 범주형이 살아 있는 데이터
# - 그룹별 boxplot, barplot, churn rate 비교용
eda_df.info()

In [ ]:
# final_df
# - 인코딩 완료된 모델 입력용 데이터
# - 상관계수, 모델링, feature importance용
final_df.info()

In [ ]:
# location 컬럼을 기준으로 그룹화하여 전체 고객 수와 이탈 고객 수 계산
state_churn_analysis = music_df.groupby('location')['churned'].agg(['count', 'sum']).reset_index()

# 컬럼명 변경
state_churn_analysis.columns = ['state', 'total_customers', 'churned_customers']

# 이탈률 계산
# 이탈률 = (이탈 고객 수 / 전체 고객 수)
state_churn_analysis['churn_rate'] = state_churn_analysis['churned_customers'] / state_churn_analysis['total_customers']

# 이탈률을 퍼센트로 변환하고 소수점 둘째 자리까지 반올림
state_churn_analysis['churn_rate_pct'] = round(state_churn_analysis['churn_rate'] * 100, 2)

# 이탈률이 높은 순서대로 정렬
state_churn_analysis = state_churn_analysis.sort_values(by='churn_rate_pct', ascending=False)

# 결과 출력
print("State별 이탈률 분석 결과:")
display(state_churn_analysis.head(10))

In [ ]:
!pip install folium

In [ ]:
# 공백 제거 및 첫 글자만 대문자로 통일
state_churn_analysis['state'] = state_churn_analysis['state'].str.strip().str.title()

In [ ]:
# 미국 50개 주 전체 리스트
us_states_full = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut',
    'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa',
    'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan',
    'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
    'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio',
    'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota',
    'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia',
    'Wisconsin', 'Wyoming'
]

# 데이터에 없는 주 출력
missing_states = [s for s in us_states_full if s not in state_churn_analysis['state'].values]
print("데이터에 없어 색이 안 칠해지는 주:", missing_states)

In [ ]:
import folium
import json
import requests

# 1. 미국 주 경계 데이터 로드
url = 'https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/us-states.json'
state_geo = requests.get(url).json()

# 주요 State의 위도/경도 좌표
state_coords = {
    'California': [36.7783, -119.4179], 'Texas': [31.9686, -99.9018],
    'Florida': [27.6648, -81.5158], 'New York': [40.7128, -74.0060],
    'Pennsylvania': [41.2033, -77.1945], 'Illinois': [40.6331, -89.3985],
    'Ohio': [40.4173, -82.9071], 'Georgia': [32.1656, -82.9001],
    'North Carolina': [35.7596, -79.0193], 'Michigan': [44.3148, -85.6024],
    'Washington': [47.7511, -120.7401], 'Arizona': [34.0489, -111.0937],
    'Massachusetts': [42.4072, -71.3824], 'Tennessee': [35.5175, -86.5804],
    'Indiana': [40.2672, -86.1349], 'Missouri': [37.9643, -91.8318],
    'Maryland': [39.0458, -76.6413], 'Wisconsin': [43.7844, -88.7879],
    'Colorado': [39.5501, -105.7821]
}

# 지도 객체 생성
m = folium.Map(location=[37.0902, -95.7129], zoom_start=4)

# 단계구분도 그리기
folium.Choropleth(
    geo_data=state_geo,
    data=state_churn_analysis,
    columns=['state', 'churn_rate_pct'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    nan_fill_color='lightgrey',
    legend_name='Customer Churn Rate (%)'
).add_to(m)

# 이탈률 TOP 3 지역에 마커 추가
top_3_states = state_churn_analysis.head(3)

for index, row in top_3_states.iterrows():
    state_name = row['state']
    churn_val = row['churn_rate_pct']

    # 좌표 데이터가 있는 경우에만 지도 위에 마커핀를 찍음
    if state_name in state_coords:
        folium.Marker(
            location=state_coords[state_name],
            popup=f"위험 지역: {state_name} ({churn_val}%)",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)

    # 콘솔에도 정보 출력
    print(f"위험 지역 강조: {state_name} ({churn_val}%)")

# 6. 최종 지도 출력
display(m)